In [ ]:
#!/usr/bin/env python3
import os
import time
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH         = "/kaggle/input/datasets/syedmdnafissameen/medqa-new/bangla-med-qa.csv"  # Adjust input path if needed
MODEL            = "md-nishat-008/TigerLLM-9B-it"
OUTPUT_DIR       = "/kaggle/working/bangla_med_benchmark"
NUM_SAMPLES      = None              # None = full dataset
CHECKPOINT_EVERY = 20

VALID_LETTERS = ["A", "B", "C", "D"]

SYSTEM_PROMPT = """You are answering a multiple-choice question. Output only the correct option letter(s). If there is one correct answer, output exactly one of A, B, C, or D. If there are multiple correct answers, output the letters in alphabetical order separated by commas (e.g., A,B or B,D or A,B,C). Output nothing else. Never give blank output."""

USER_TEMPLATE = """Question: {question}
A) {opt_a}
B) {opt_b}
C) {opt_c}
D) {opt_d}
Answer:"""


# ── Label helpers ─────────────────────────────────────────────────────────────
def normalize_label(raw):
    """Extract and sort all A/B/C/D letters into a canonical string e.g. 'AB', 'ACD'."""
    if raw is None:
        return None
    found = sorted(set(ch for ch in str(raw).upper() if ch in VALID_LETTERS))
    return "".join(found) if found else None


def parse_response(raw_text):
    return normalize_label(raw_text)


def build_user_message(row):
    return USER_TEMPLATE.format(
        question=row["question"],
        opt_a=row["options/A"],
        opt_b=row["options/B"],
        opt_c=row["options/C"],
        opt_d=row["options/D"],
    )


# ── Local HF Model Initialization & Call ──────────────────────────────────────
def load_local_model(model_name):
    print("Loading 8-bit quantized model...")
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,   # <-- was float16, causes garbage on Gemma-based models
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def _build_prompt(tokenizer, user_message):
    # Some chat templates don't support a separate "system" role, so we
    # fall back to merging it into the user turn if that happens.
    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        messages = [
            {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_message},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


def call_local_model(model, tokenizer, user_message):
    start = time.monotonic()

    try:
        prompt = _build_prompt(tokenizer, user_message)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Slice generated tokens to get response only
        generated_tokens = output_ids[0][inputs.input_ids.shape[-1]:]
        content = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        latency = time.monotonic() - start

        return content, None, latency

    except Exception as e:
        latency = time.monotonic() - start
        return None, f"execution_error:{e}", latency


# ── Checkpoint ────────────────────────────────────────────────────────────────
def checkpoint_path(out_dir, model):
    return os.path.join(out_dir, f"checkpoint_{model.replace('/', '__')}.csv")

def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    return pd.read_csv(path) if os.path.exists(path) else None

def save_checkpoint(out_dir, model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(out_dir, model), index=False)


# ── Main loop ─────────────────────────────────────────────────────────────────
def run_model_on_dataset(model_obj, tokenizer, model_name, df, out_dir):
    existing  = load_checkpoint(out_dir, model_name)
    rows      = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows)

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows

    pbar = tqdm(range(start_idx, len(df)), desc=model_name, unit="row",
                initial=start_idx, total=len(df))

    for i in pbar:
        row      = df.iloc[i]
        label    = normalize_label(row["answer"])
        user_msg = build_user_message(row)

        raw, err, latency = call_local_model(model_obj, tokenizer, user_msg)
        pred = parse_response(raw)

        retries = 0
        while pred is None and retries < 3:
            print(f"[{i}] pred=None (raw='{raw}', err='{err}') — retrying…")
            raw, err, latency = call_local_model(model_obj, tokenizer, user_msg)
            pred = parse_response(raw)
            retries += 1

        is_multi  = len(label) > 1 if label else False
        correct   = (pred == label) if pred is not None else False

        print(f"[{i}] raw='{raw}' → pred={pred} | label={label} | multi={is_multi} | correct={correct}")

        rows.append({
            "serial_no":    row.get("serial_no", i),
            "exam_name":    row.get("exam_name", ""),
            "question":     row["question"],
            "label":        label,
            "prediction":   pred,
            "is_multi":     is_multi,
            "correct":      correct,
            "latency":      latency,
            "raw_response": raw,
            "error":        err,
        })

        if (i - start_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(out_dir, model_name, rows)

    save_checkpoint(out_dir, model_name, rows)
    return rows


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(rows):
    """Compute metrics overall, and separately for single vs multi-answer rows."""

    def _metrics(subset):
        valid = [r for r in subset if r["prediction"] is not None and str(r["prediction"]).strip() not in ("", "nan")]
        if not valid:
            return {"accuracy": None, "precision": None, "recall": None, "f1": None, "valid": 0, "total": len(subset)}
        yt = [str(r["label"])      for r in valid]
        yp = [str(r["prediction"]) for r in valid]
        classes = sorted(set(yt) | set(yp))
        return {
            "accuracy":  accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, labels=classes, average="macro", zero_division=0),
            "recall":    recall_score(yt, yp,    labels=classes, average="macro", zero_division=0),
            "f1":        f1_score(yt, yp,        labels=classes, average="macro", zero_division=0),
            "valid":     len(valid),
            "total":     len(subset),
        }

    latencies    = [r["latency"] for r in rows if r["latency"] is not None]
    single_rows  = [r for r in rows if not r["is_multi"]]
    multi_rows   = [r for r in rows if r["is_multi"]]

    return {
        "overall":       _metrics(rows),
        "single_answer": _metrics(single_rows),
        "multi_answer":  _metrics(multi_rows),
        "avg_latency":   sum(latencies) / len(latencies) if latencies else None,
    }


# ── Entry point ───────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    required = {"question", "options/A", "options/B", "options/C", "options/D", "answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required}")

    df["answer"] = df["answer"].apply(normalize_label)

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"Dataset: {len(df)} rows")
    print(f"Single-answer: {(df['answer'].str.len() == 1).sum()} | Multi-answer: {(df['answer'].str.len() > 1).sum()}")

    # Initialize local quantized model
    model_obj, tokenizer = load_local_model(MODEL)

    rows    = run_model_on_dataset(model_obj, tokenizer, MODEL, df, OUTPUT_DIR)
    metrics = compute_metrics(rows)

    # ── Save predictions ──
    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "predictions.csv"), index=False, encoding="utf-8-sig")

    # ── Save results ──
    results_rows = []
    for split, m in metrics.items():
        if split == "avg_latency":
            continue
        results_rows.append({
            "model":       MODEL,
            "split":       split,
            "accuracy":    m.get("accuracy"),
            "precision":   m.get("precision"),
            "recall":      m.get("recall"),
            "f1":          m.get("f1"),
            "valid":       m.get("valid"),
            "total":       m.get("total"),
            "avg_latency": metrics["avg_latency"],
        })

    results_df = pd.DataFrame(results_rows)
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "benchmark_results.csv"), index=False, encoding="utf-8-sig")

    print("\n── Results ──")
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()#!/usr/bin/env python3
import os
import time
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH         = ""  # Adjust input path if needed
MODEL            = "md-nishat-008/TigerLLM-9B-it"
OUTPUT_DIR       = "/kaggle/working/bangla_med_benchmark"
NUM_SAMPLES      = None              # None = full dataset
CHECKPOINT_EVERY = 20

VALID_LETTERS = ["A", "B", "C", "D"]

SYSTEM_PROMPT = """You are answering a multiple-choice question. Output only the correct option letter(s). If there is one correct answer, output exactly one of A, B, C, or D. If there are multiple correct answers, output the letters in alphabetical order separated by commas (e.g., A,B or B,D or A,B,C). Output nothing else. Never give blank output."""

USER_TEMPLATE = """Question: {question}
A) {opt_a}
B) {opt_b}
C) {opt_c}
D) {opt_d}
Answer:"""


# ── Label helpers ─────────────────────────────────────────────────────────────
def normalize_label(raw):
    """Extract and sort all A/B/C/D letters into a canonical string e.g. 'AB', 'ACD'."""
    if raw is None:
        return None
    found = sorted(set(ch for ch in str(raw).upper() if ch in VALID_LETTERS))
    return "".join(found) if found else None


def parse_response(raw_text):
    return normalize_label(raw_text)


def build_user_message(row):
    return USER_TEMPLATE.format(
        question=row["question"],
        opt_a=row["options/A"],
        opt_b=row["options/B"],
        opt_c=row["options/C"],
        opt_d=row["options/D"],
    )


# ── Local HF Model Initialization & Call ──────────────────────────────────────
def load_local_model(model_name):
    print("Loading 8-bit quantized model...")
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def _build_prompt(tokenizer, user_message):
    # Some chat templates don't support a separate "system" role, so we
    # fall back to merging it into the user turn if that happens.
    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        messages = [
            {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_message},
        ]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


def call_local_model(model, tokenizer, user_message):
    start = time.monotonic()

    try:
        prompt = _build_prompt(tokenizer, user_message)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Slice generated tokens to get response only
        generated_tokens = output_ids[0][inputs.input_ids.shape[-1]:]
        content = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        latency = time.monotonic() - start

        return content, None, latency

    except Exception as e:
        latency = time.monotonic() - start
        return None, f"execution_error:{e}", latency


# ── Checkpoint ────────────────────────────────────────────────────────────────
def checkpoint_path(out_dir, model):
    return os.path.join(out_dir, f"checkpoint_{model.replace('/', '__')}.csv")

def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    return pd.read_csv(path) if os.path.exists(path) else None

def save_checkpoint(out_dir, model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(out_dir, model), index=False)


# ── Main loop ─────────────────────────────────────────────────────────────────
def run_model_on_dataset(model_obj, tokenizer, model_name, df, out_dir):
    existing  = load_checkpoint(out_dir, model_name)
    rows      = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows)

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows

    pbar = tqdm(range(start_idx, len(df)), desc=model_name, unit="row",
                initial=start_idx, total=len(df))

    for i in pbar:
        row      = df.iloc[i]
        label    = normalize_label(row["answer"])
        user_msg = build_user_message(row)

        raw, err, latency = call_local_model(model_obj, tokenizer, user_msg)
        pred = parse_response(raw)

        retries = 0
        while pred is None and retries < 3:
            print(f"[{i}] pred=None (raw='{raw}', err='{err}') — retrying…")
            raw, err, latency = call_local_model(model_obj, tokenizer, user_msg)
            pred = parse_response(raw)
            retries += 1

        is_multi  = len(label) > 1 if label else False
        correct   = (pred == label) if pred is not None else False

        print(f"[{i}] raw='{raw}' → pred={pred} | label={label} | multi={is_multi} | correct={correct}")

        rows.append({
            "serial_no":    row.get("serial_no", i),
            "exam_name":    row.get("exam_name", ""),
            "question":     row["question"],
            "label":        label,
            "prediction":   pred,
            "is_multi":     is_multi,
            "correct":      correct,
            "latency":      latency,
            "raw_response": raw,
            "error":        err,
        })

        if (i - start_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(out_dir, model_name, rows)

    save_checkpoint(out_dir, model_name, rows)
    return rows


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(rows):
    """Compute metrics overall, and separately for single vs multi-answer rows."""

    def _metrics(subset):
        valid = [r for r in subset if r["prediction"] is not None and str(r["prediction"]).strip() not in ("", "nan")]
        if not valid:
            return {"accuracy": None, "precision": None, "recall": None, "f1": None, "valid": 0, "total": len(subset)}
        yt = [str(r["label"])      for r in valid]
        yp = [str(r["prediction"]) for r in valid]
        classes = sorted(set(yt) | set(yp))
        return {
            "accuracy":  accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, labels=classes, average="macro", zero_division=0),
            "recall":    recall_score(yt, yp,    labels=classes, average="macro", zero_division=0),
            "f1":        f1_score(yt, yp,        labels=classes, average="macro", zero_division=0),
            "valid":     len(valid),
            "total":     len(subset),
        }

    latencies    = [r["latency"] for r in rows if r["latency"] is not None]
    single_rows  = [r for r in rows if not r["is_multi"]]
    multi_rows   = [r for r in rows if r["is_multi"]]

    return {
        "overall":       _metrics(rows),
        "single_answer": _metrics(single_rows),
        "multi_answer":  _metrics(multi_rows),
        "avg_latency":   sum(latencies) / len(latencies) if latencies else None,
    }


# ── Entry point ───────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    required = {"question", "options/A", "options/B", "options/C", "options/D", "answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required}")

    df["answer"] = df["answer"].apply(normalize_label)

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"Dataset: {len(df)} rows")
    print(f"Single-answer: {(df['answer'].str.len() == 1).sum()} | Multi-answer: {(df['answer'].str.len() > 1).sum()}")

    # Initialize local quantized model
    model_obj, tokenizer = load_local_model(MODEL)

    rows    = run_model_on_dataset(model_obj, tokenizer, MODEL, df, OUTPUT_DIR)
    metrics = compute_metrics(rows)

    # ── Save predictions ──
    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "predictions.csv"), index=False, encoding="utf-8-sig")

    # ── Save results ──
    results_rows = []
    for split, m in metrics.items():
        if split == "avg_latency":
            continue
        results_rows.append({
            "model":       MODEL,
            "split":       split,
            "accuracy":    m.get("accuracy"),
            "precision":   m.get("precision"),
            "recall":      m.get("recall"),
            "f1":          m.get("f1"),
            "valid":       m.get("valid"),
            "total":       m.get("total"),
            "avg_latency": metrics["avg_latency"],
        })

    results_df = pd.DataFrame(results_rows)
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "benchmark_results.csv"), index=False, encoding="utf-8-sig")

    print("\n── Results ──")
    print(results_df.to_string(index=False))


if __name__ == "__main__":
    main()